<a href="https://colab.research.google.com/github/shahddroubi/Driver-Behavior/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###data1

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.neural_network import MLPClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Conv1D, LSTM, Flatten, Dropout, BatchNormalization, MaxPooling1D

# 1. LOAD DATA
train = pd.read_csv("/content/drive/MyDrive/train_motion_data.csv")
test  = pd.read_csv("/content/drive/MyDrive/test_motion_data.csv")

train['Class'] = train['Class'].replace('SLOW','CONSERVATIVE')
test['Class']  = test['Class'].replace('SLOW','CONSERVATIVE')


time_train = train['Timestamp'].values
time_test  = test['Timestamp'].values

train.drop(columns=['Timestamp'],errors='ignore',inplace=True)
test.drop(columns=['Timestamp'],errors='ignore',inplace=True)

full_data = pd.concat([train,test],ignore_index=True)
print("Total Samples:",len(full_data))


# 2. ROAD TYPE SPLIT

full_data['Acc_Mag']=np.sqrt(full_data['AccX']**2 + full_data['AccY']**2 + full_data['AccZ']**2)
full_data['Gyro_Mag']=np.sqrt(full_data['GyroX']**2 + full_data['GyroY']**2 + full_data['GyroZ']**2)
full_data['Dynamic_Index']=full_data['Acc_Mag'] + full_data['Gyro_Mag']

threshold = full_data['Dynamic_Index'].quantile(0.30)
intersections_df = full_data[full_data['Dynamic_Index']<=threshold]
highways_df = full_data[full_data['Dynamic_Index']>threshold]

print("Dynamic Threshold:",threshold)
print("Intersections:",len(intersections_df))
print("Highways:",len(highways_df))


# 3. FEATURE EXTRACTION

features = ['AccX','AccY','AccZ','GyroX','GyroY','GyroZ']

def extract_features(df,window_size=15):
    X=[]
    y=[]
    for i in range(0,len(df)-window_size,window_size):
        window=df.iloc[i:i+window_size]
        stats=[]
        for col in features:
            signal = window[col].values
            stats.extend([np.mean(signal), np.std(signal), np.max(signal), np.min(signal)])
            stats.append(np.sum(signal**2)/len(signal))  # energy
            jerk = np.diff(signal)
            stats.append(np.mean(np.abs(jerk)))  # jerk
            hist,_ = np.histogram(signal,bins=10)
            prob = hist/(np.sum(hist)+1e-6)
            stats.append(-np.sum(prob*np.log(prob+1e-6)))  # entropy
        X.append(stats)
        y.append(window['Class'].mode()[0])
    return np.array(X), np.array(y)

X_int, y_int = extract_features(intersections_df)
X_high, y_high = extract_features(highways_df)

# 4. SCALING + ENCODING

scaler_int = StandardScaler()
scaler_high = StandardScaler()

X_int_scaled = scaler_int.fit_transform(X_int)
X_high_scaled = scaler_high.fit_transform(X_high)

le_int = LabelEncoder()
le_high = LabelEncoder()

y_int_encoded = le_int.fit_transform(y_int)
y_high_encoded = le_high.fit_transform(y_high)

# 5. SELECT TOP 5 FEATURES USING RANDOM FOREST
def top_features(X,y,feature_names,n_top=5):
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X,y)
    importances = rf.feature_importances_
    idx = np.argsort(importances)[::-1][:n_top]
    top_feats = [(feature_names[i], importances[i]) for i in idx]
    return idx, top_feats

feature_names = []
for f in features:
    feature_names.extend([f+"_mean", f+"_std", f+"_max", f+"_min", f+"_energy", f+"_jerk", f+"_entropy"])

idx_int, top_int_feats = top_features(X_int_scaled,y_int_encoded,feature_names)
idx_high, top_high_feats = top_features(X_high_scaled,y_high_encoded,feature_names)

print("Top 5 Intersection Features:", top_int_feats)
print("Top 5 Highway Features:", top_high_feats)


X_int_selected = X_int_scaled[:, idx_int]
X_high_selected = X_high_scaled[:, idx_high]

# 6. ML MODELS
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf',C=5,probability=True,gamma = 'scale'),
    "KNN": KNeighborsClassifier(n_neighbors=5, metric = 'minkowski'),
    "XGBoost": XGBClassifier(n_estimators=100,eval_metric='mlogloss', random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', solver='adam', max_iter=300, random_state=42)

}
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize

def run_ml(X_train, X_test, y_train, y_test, le, road_type):

    print(f"\n===== {road_type} ML RESULTS =====")

    for name, model in models.items():

        model.fit(X_train,y_train)

        pred = model.predict(X_test)

        if hasattr(model,"predict_proba"):
            prob = model.predict_proba(X_test)
        else:
            prob = None

        # Metrics
        acc = accuracy_score(y_test,pred)
        prec = precision_score(y_test,pred,average='weighted')
        rec = recall_score(y_test,pred,average='weighted')
        f1 = f1_score(y_test,pred,average='weighted')

        # ROC AUC (multi-class)
        if prob is not None:
            y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
            roc = roc_auc_score(y_test_bin, prob, multi_class='ovr')
        else:
            roc = "N/A"

        print(f"\n{name} Results ({road_type})")
        print("Accuracy:","{:.4f}".format(acc))
        print("Precision:","{:.4f}".format(prec))
        print("Recall:","{:.4f}".format(rec))
        print("F1 Score:","{:.4f}".format(f1))
        print("ROC AUC:", "{:.4f}".format(roc) if isinstance(roc, float) else roc)

        print("\nClassification Report:")
        print(classification_report(y_test,pred,target_names=le.classes_))

        cm = confusion_matrix(y_test, pred)

        plt.figure(figsize=(6,5))
        sns.heatmap(cm,
                    annot=True,
                    fmt='d',
                    cmap='Blues',
                    xticklabels=le.classes_,
                    yticklabels=le.classes_,
                    annot_kws={"size": 18, "weight": "bold"})

        plt.title(f"{name} Confusion Matrix ({road_type})", fontsize=14)
        plt.xlabel("Predicted", fontsize=12)
        plt.ylabel("Actual", fontsize=12)
        plt.xticks(fontsize=11)
        plt.yticks(fontsize=11)
        plt.show()

run_ml(*train_test_split(X_int_selected,y_int_encoded,test_size=0.2,random_state=42,stratify=y_int_encoded),
       le_int,"Intersections")
run_ml(*train_test_split(X_high_selected,y_high_encoded,test_size=0.2,random_state=42,stratify=y_high_encoded),
       le_high,"Highways")

from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(models, X, y, road_type):

    print(f"\n===== {road_type} CROSS VALIDATION RESULTS =====")

    for name, model in models.items():

        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

        print(f"\n{name}")
        print("CV Accuracy Mean:", scores.mean())
        print("CV Accuracy Std:", scores.std())

run_cv(models, X_int_selected, y_int_encoded, "Intersections")
run_cv(models, X_high_selected, y_high_encoded, "Highways")
# 8. DL MODELS
def create_cnn_lstm(input_shape,n_classes):
    # CNN
    model = Sequential()
    model.add(Conv1D(32,3,activation='relu',input_shape=input_shape))
    model.add(Dropout(0.5))
    model.add(BatchNormalization())
    model.add(Flatten())
    model.add(Dense(64,activation='relu'))
    model.add(Dense(n_classes,activation='softmax'))
    model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
    return model

def create_lstm(input_shape,n_classes):
    model = Sequential()
    model.add(LSTM(64,input_shape=input_shape))
    model.add(Dense(64,activation='relu'))
    model.add(Dense(n_classes,activation='softmax'))
    model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
    return model

# Prepare sequences for DL
def prepare_dl_data(X, y):

    # كل sample = sequence بطول 5 features
    X_seq = X.reshape(X.shape[0], X.shape[1], 1)

    y_cat = to_categorical(y)

    return X_seq, y_cat


X_seq_int, y_seq_int_cat = prepare_dl_data(X_int_selected,y_int_encoded)
X_seq_high, y_seq_high_cat = prepare_dl_data(X_high_selected,y_high_encoded)
def train_dl_model(X, y, road_type):

    n_classes = y.shape[1]

    print(f"\n===== {road_type} DL RESULTS =====")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=np.argmax(y, axis=1)
    )

    models_dl = {
        "1D-CNN": create_cnn_lstm(
            (X.shape[1], X.shape[2]),
            n_classes
        ),

        "LSTM": create_lstm(
            (X.shape[1], X.shape[2]),
            n_classes
        )
    }

    for name, model in models_dl.items():

        history = model.fit(
            X_train,
            y_train,
            epochs=15,
            batch_size=8,
            validation_data=(X_test, y_test),
            verbose=0
        )

        loss, acc = model.evaluate(
            X_test,
            y_test,
            verbose=0
        )

        print(f"{name} Test Accuracy ({road_type}): {acc:.4f}")
        print(f"{name} Test Loss ({road_type}): {loss:.4f}")


train_dl_model(X_seq_int, y_seq_int_cat,"Intersections")
train_dl_model(X_seq_high, y_seq_high_cat,"Highways")
from sklearn.model_selection import StratifiedKFold

def dl_cross_validation(X, y, road_type, model_type="CNN"):

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    acc_scores = []

    for train_idx, test_idx in skf.split(X, np.argmax(y, axis=1)):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if model_type == "CNN":
            model = create_cnn_lstm((X.shape[1], X.shape[2]), y.shape[1])
        else:
            model = create_lstm((X.shape[1], X.shape[2]), y.shape[1])

        model.fit(X_train, y_train, epochs=10, batch_size=8, verbose=0)

        loss, acc = model.evaluate(X_test, y_test, verbose=0)

        acc_scores.append(acc)

    print(f"\n{model_type} CV Accuracy ({road_type})")
    print("Mean:", np.mean(acc_scores))
    print("Std:", np.std(acc_scores))
dl_cross_validation(X_seq_int, y_seq_int_cat, "Intersections", "CNN")
dl_cross_validation(X_seq_high, y_seq_high_cat, "Highways", "CNN")
dl_cross_validation(X_seq_int, y_seq_int_cat, "Intersections", "LSTM")
dl_cross_validation(X_seq_high, y_seq_high_cat, "Highways", "LSTM")
def plot_features(top_feats,road_type):
    names = [f[0] for f in top_feats]
    values = [f[1] for f in top_feats]
    plt.figure(figsize=(8,5))
    sns.barplot(x=names,y=values)
    plt.title(f"Top 5 Feature Importances ({road_type})")
    plt.xticks(rotation=45)
    plt.show()

plot_features(top_int_feats,"Intersections")
plot_features(top_high_feats,"Highways")



In [ ]:
# ==============================
# IMPORTS oversample
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.neural_network import MLPClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Conv1D, LSTM, Flatten, Dropout, BatchNormalization, MaxPooling1D


# 1. LOAD DATA
train = pd.read_csv("/content/drive/MyDrive/train_motion_data.csv")
test  = pd.read_csv("/content/drive/MyDrive/test_motion_data.csv")

train['Class'] = train['Class'].replace('SLOW','CONSERVATIVE')
test['Class']  = test['Class'].replace('SLOW','CONSERVATIVE')

time_train = train['Timestamp'].values
time_test  = test['Timestamp'].values

train.drop(columns=['Timestamp'],errors='ignore',inplace=True)
test.drop(columns=['Timestamp'],errors='ignore',inplace=True)

full_data = pd.concat([train,test],ignore_index=True)
print("Total Samples:",len(full_data))

# 2. ROAD TYPE SPLIT
full_data['Acc_Mag']=np.sqrt(full_data['AccX']**2 + full_data['AccY']**2 + full_data['AccZ']**2)
full_data['Gyro_Mag']=np.sqrt(full_data['GyroX']**2 + full_data['GyroY']**2 + full_data['GyroZ']**2)
full_data['Dynamic_Index']=full_data['Acc_Mag'] + full_data['Gyro_Mag']

threshold = full_data['Dynamic_Index'].quantile(0.30)
intersections_df = full_data[full_data['Dynamic_Index']<=threshold]
highways_df = full_data[full_data['Dynamic_Index']>threshold]

print("Dynamic Threshold:",threshold)
print("Intersections:",len(intersections_df))
print("Highways:",len(highways_df))

# 3. FEATURE EXTRACTION (STAT+ENERGY+JERK+ENTROPY)

features = ['AccX','AccY','AccZ','GyroX','GyroY','GyroZ']

def extract_features(df,window_size=15):
    X=[]
    y=[]
    for i in range(0,len(df)-window_size,window_size):
        window=df.iloc[i:i+window_size]
        stats=[]
        for col in features:
            signal = window[col].values
            stats.extend([np.mean(signal), np.std(signal), np.max(signal), np.min(signal)])
            stats.append(np.sum(signal**2)/len(signal))  # energy
            jerk = np.diff(signal)
            stats.append(np.mean(np.abs(jerk)))  # jerk
            hist,_ = np.histogram(signal,bins=10)
            prob = hist/(np.sum(hist)+1e-6)
            stats.append(-np.sum(prob*np.log(prob+1e-6)))  # entropy
        X.append(stats)
        y.append(window['Class'].mode()[0])
    return np.array(X), np.array(y)

X_int, y_int = extract_features(intersections_df)
X_high, y_high = extract_features(highways_df)

# 4. SCALING + ENCODING

scaler_int = StandardScaler()
scaler_high = StandardScaler()

X_int_scaled = scaler_int.fit_transform(X_int)
X_high_scaled = scaler_high.fit_transform(X_high)

le_int = LabelEncoder()
le_high = LabelEncoder()

y_int_encoded = le_int.fit_transform(y_int)
y_high_encoded = le_high.fit_transform(y_high)

# 5. SELECT TOP 5 FEATURES USING RANDOM FOREST

def top_features(X,y,feature_names,n_top=5):
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X,y)
    importances = rf.feature_importances_
    idx = np.argsort(importances)[::-1][:n_top]
    top_feats = [(feature_names[i], importances[i]) for i in idx]
    return idx, top_feats

feature_names = []
for f in features:
    feature_names.extend([f+"_mean", f+"_std", f+"_max", f+"_min", f+"_energy", f+"_jerk", f+"_entropy"])

idx_int, top_int_feats = top_features(X_int_scaled,y_int_encoded,feature_names)
idx_high, top_high_feats = top_features(X_high_scaled,y_high_encoded,feature_names)

print("Top 5 Intersection Features:", top_int_feats)
print("Top 5 Highway Features:", top_high_feats)

# Use same 5 features for ML and DL
X_int_selected = X_int_scaled[:, idx_int]
X_high_selected = X_high_scaled[:, idx_high]
# ==============================
# BALANCE CLASSES USING SMOTE
# ==============================
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)

# Intersections
X_int_bal, y_int_bal = sm.fit_resample(X_int_selected, y_int_encoded)
print("Intersections - Original counts:", np.bincount(y_int_encoded))
print("Intersections - Resampled counts:", np.bincount(y_int_bal))

# Highways
X_high_bal, y_high_bal = sm.fit_resample(X_high_selected, y_high_encoded)
print("Highways - Original counts:", np.bincount(y_high_encoded))
print("Highways - Resampled counts:", np.bincount(y_high_bal))

# 6. ML MODELS

models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf',C=5,probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5, metric = 'minkowski'),
    "XGBoost": XGBClassifier(n_estimators=100,eval_metric='mlogloss', random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', solver='adam', max_iter=300, random_state=42)

}
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize

def run_ml(X_train, X_test, y_train, y_test, le, road_type):

    print(f"\n===== {road_type} ML RESULTS =====")

    for name, model in models.items():

        model.fit(X_train,y_train)

        pred = model.predict(X_test)


        if hasattr(model,"predict_proba"):
            prob = model.predict_proba(X_test)
        else:
            prob = None

        # Metrics
        acc = accuracy_score(y_test,pred)
        prec = precision_score(y_test,pred,average='weighted')
        rec = recall_score(y_test,pred,average='weighted')
        f1 = f1_score(y_test,pred,average='weighted')

        # ROC AUC (multi-class)
        if prob is not None:
            y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
            roc = roc_auc_score(y_test_bin, prob, multi_class='ovr')
        else:
            roc = "N/A"

        print(f"\n{name} Results ({road_type})")
        print("Accuracy:","{:.4f}".format(acc))
        print("Precision:","{:.4f}".format(prec))
        print("Recall:","{:.4f}".format(rec))
        print("F1 Score:","{:.4f}".format(f1))
        print("ROC AUC:", "{:.4f}".format(roc) if isinstance(roc, float) else roc)
        print("\nClassification Report:")
        print(classification_report(y_test,pred,target_names=le.classes_))


        cm = confusion_matrix(y_test, pred)

        plt.figure(figsize=(6,5))
        sns.heatmap(cm,
                    annot=True,
                    fmt='d',
                    cmap='Blues',
                    xticklabels=le.classes_,
                    yticklabels=le.classes_,
                    annot_kws={"size": 18, "weight": "bold"})
        plt.title(f"{name} Confusion Matrix ({road_type})", fontsize=14)
        plt.xlabel("Predicted", fontsize=12)
        plt.ylabel("Actual", fontsize=12)
        plt.xticks(fontsize=11)
        plt.yticks(fontsize=11)
        plt.show()
run_ml(*train_test_split(X_int_bal, y_int_bal, test_size=0.2, random_state=42, stratify=y_int_bal),
       le_int, "Intersections")

run_ml(*train_test_split(X_high_bal, y_high_bal, test_size=0.2, random_state=42, stratify=y_high_bal),
       le_high, "Highways")


# 8. DL MODELS

def create_cnn_lstm(input_shape,n_classes):
    # CNN
    model = Sequential()
    model.add(Conv1D(32,3,activation='relu',input_shape=input_shape))
    model.add(Dropout(0.5))
    model.add(BatchNormalization())
    model.add(Flatten())
    model.add(Dense(64,activation='relu'))
    model.add(Dense(n_classes,activation='softmax'))
    model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
    return model

def create_lstm(input_shape,n_classes):
    model = Sequential()
    model.add(LSTM(64,input_shape=input_shape))
    model.add(Dense(64,activation='relu'))
    model.add(Dense(n_classes,activation='softmax'))
    model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
    return model


def prepare_dl_data(X, y):

    X_seq = X.reshape(X.shape[0], X.shape[1], 1)

    y_cat = to_categorical(y)

    return X_seq, y_cat
X_seq_int_bal, y_seq_int_bal_cat = prepare_dl_data(X_int_bal, y_int_bal)
X_seq_high_bal, y_seq_high_bal_cat = prepare_dl_data(X_high_bal, y_high_bal)

def train_dl_model(X, y, road_type):

    n_classes = y.shape[1]

    print(f"\n===== {road_type} DL RESULTS =====")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=np.argmax(y, axis=1)
    )

    models_dl = {
        "1D-CNN": create_cnn_lstm(
            (X.shape[1], X.shape[2]),
            n_classes
        ),

        "LSTM": create_lstm(
            (X.shape[1], X.shape[2]),
            n_classes
        )
    }

    for name, model in models_dl.items():

        history = model.fit(
            X_train,
            y_train,
            epochs=15,
            batch_size=8,
            validation_data=(X_test, y_test),
            verbose=0
        )

        loss, acc = model.evaluate(
            X_test,
            y_test,
            verbose=0
        )

        print(f"{name} Test Accuracy ({road_type}): {acc:.4f}")
        print(f"{name} Test Loss ({road_type}): {loss:.4f}")

train_dl_model(X_seq_int_bal, y_seq_int_bal_cat, "Intersections")
train_dl_model(X_seq_high_bal, y_seq_high_bal_cat, "Highways")


def plot_features(top_feats,road_type):
    names = [f[0] for f in top_feats]
    values = [f[1] for f in top_feats]
    plt.figure(figsize=(8,5))
    sns.barplot(x=names,y=values)
    plt.title(f"Top 5 Feature Importances ({road_type})")
    plt.xticks(rotation=45)
    plt.show()

plot_features(top_int_feats,"Intersections")
plot_features(top_high_feats,"Highways")



In [ ]:
###data2
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# تحميل البيانات
data = pd.read_csv('/content/drive/MyDrive/Driving Data(KIA SOUL)_(150728-160714)_(10 Drivers_A-J).csv')

numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns.tolist()
exclude_cols = ['Time(s)', 'class']
numeric_cols = [col for col in numeric_cols if col not in exclude_cols]

data_numeric = data[numeric_cols].fillna(0)

# Standardization
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_numeric)

# PCA
pca = PCA(n_components=5)
pca.fit(data_scaled)

# أهمية كل feature
importance = pd.DataFrame(pca.components_.T, index=numeric_cols, columns=[f'PC{i+1}' for i in range(5)])
importance['PC1_PC2'] = importance['PC1'].abs() + importance['PC2'].abs()

importance_sorted = importance.sort_values('PC1_PC2', ascending=False)

print("Top important features based on PCA (PC1 + PC2):")
print(importance_sorted[['PC1_PC2']])
import matplotlib.pyplot as plt
import seaborn as sns

# نأخذ أهم 10 أعمدة حسب PC1 + PC2
top_features = importance_sorted.head(10).reset_index()

plt.figure(figsize=(10,6))
sns.barplot(data=top_features, x='PC1_PC2', y='index', palette='viridis')
plt.xlabel('Importance (PC1 + PC2)')
plt.ylabel('Feature')
plt.title('Top 10 Most Important Features Based on PCA')
plt.show()
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
# تحميل البيانات
data = pd.read_csv('/content/drive/MyDrive/Driving Data(KIA SOUL)_(150728-160714)_(10 Drivers_A-J).csv')

# الأعمدة المختارة
important_features = [
    'Engine_torque',
    'Engine_torque_after_correction',
    'Calculated_LOAD_value',
    'Flywheel_torque',
    'Flywheel_torque_(after_torque_interventions)',
    'Wheel_velocity_rear_left-hand',
    'Wheel_velocity_front_left-hand',
    'Wheel_velocity_rear_right-hand',
    'Vehicle_speed',
    'Wheel_velocity_front_right-hand'
]

# تقسيم البيانات
highway_data = data[(data['Vehicle_speed'] > 60) &
                    (data['Acceleration_speed_-_Longitudinal'] < 2) &
                    (data['Steering_wheel_angle'].abs() < 10)].copy()

intersection_data = data[(data['Vehicle_speed'] <= 60) |
                         (data['Acceleration_speed_-_Longitudinal'] > 2) |
                         (data['Steering_wheel_angle'].abs() > 30)].copy()
print("عدد العينات للطرق السريعة:", len(highway_data))
print("عدد العينات للتقاطعات:", len(intersection_data))
# تجهيز البيانات
highway_features = highway_data[important_features].copy().fillna(0)
intersection_features = intersection_data[important_features].copy().fillna(0)

# Standardization
scaler = StandardScaler()
highway_scaled = scaler.fit_transform(highway_features)
intersection_scaled = scaler.fit_transform(intersection_features)

# KMeans
kmeans_highway = KMeans(n_clusters=3, random_state=42)
highway_clusters = kmeans_highway.fit_predict(highway_scaled)
highway_data['Cluster'] = highway_clusters

kmeans_intersection = KMeans(n_clusters=3, random_state=42)
intersection_clusters = kmeans_intersection.fit_predict(intersection_scaled)
intersection_data['Cluster'] = intersection_clusters

# Silhouette Scores
print("Silhouette Score للطرق السريعة:", silhouette_score(highway_scaled, highway_clusters))
print("Silhouette Score للتقاطعات:", silhouette_score(intersection_scaled, intersection_clusters))
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score


# تقييم الطرق السريعة - Davies-Bouldin
db_index_highway = davies_bouldin_score(highway_scaled, highway_clusters)
print("Davies-Bouldin Index للطرق السريعة:", db_index_highway)

# تقييم التقاطعات - Davies-Bouldin
db_index_intersection = davies_bouldin_score(intersection_scaled, intersection_clusters)
print("Davies-Bouldin Index للتقاطعات:", db_index_intersection)

# تقييم الطرق السريعة - Calinski-Harabasz
ch_score_highway = calinski_harabasz_score(highway_scaled, highway_clusters)
print("Calinski-Harabasz Score للطرق السريعة:", ch_score_highway)

# تقييم التقاطعات - Calinski-Harabasz
ch_score_intersection = calinski_harabasz_score(intersection_scaled, intersection_clusters)
print("Calinski-Harabasz Score للتقاطعات:", ch_score_intersection)

# توزيع أحجام الكلاستر - الطرق السريعة
unique_highway, counts_highway = np.unique(highway_clusters, return_counts=True)
print("توزيع الكلاسترات للطرق السريعة:", dict(zip(unique_highway, counts_highway)))

# توزيع أحجام الكلاستر - التقاطعات
unique_intersection, counts_intersection = np.unique(intersection_clusters, return_counts=True)
print("توزيع الكلاسترات للتقاطعات:", dict(zip(unique_intersection, counts_intersection)))

# تحليل بصري باستخدام PCA
def plot_pca_clusters(data_scaled, clusters, title):
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(data_scaled)

    df_pca = pd.DataFrame()
    df_pca['PCA1'] = pca_result[:, 0]
    df_pca['PCA2'] = pca_result[:, 1]
    df_pca['Cluster'] = clusters

    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_pca, x='PCA1', y='PCA2', hue='Cluster', palette='Set2', s=50)
    plt.title(f' PCA  - {title}')
    plt.show()

# رسم التجميعات
plot_pca_clusters(highway_scaled, highway_clusters, "highway")
plot_pca_clusters(intersection_scaled, intersection_clusters, "intersection")
highway_data['Cluster'] = highway_clusters

cluster_profile = highway_data.groupby('Cluster')[important_features].mean()
print(cluster_profile)
intersection_data['Cluster'] = intersection_clusters

cluster_profile_inter = intersection_data.groupby('Cluster')[important_features].mean()
print(cluster_profile_inter)
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import numpy as np

# دالة لاختيار eps تلقائياً باستخدام k-distance graph
def find_eps(data_scaled, k=10):
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(data_scaled)
    distances, indices = neighbors_fit.kneighbors(data_scaled)
    # نأخذ المسافة لـ k-th nearest neighbor
    k_distances = np.sort(distances[:, k-1])
    plt.figure(figsize=(8,4))
    plt.plot(k_distances)
    plt.title(f'k-distance Graph (k={k})')
    plt.xlabel('Points sorted by distance')
    plt.ylabel(f'{k}-th nearest neighbor distance')
    plt.show()
    print("ابحث عن نقطة الانحناء لتحديد eps المناسب")
    return k_distances
highway_k_dist = find_eps(highway_scaled, k=10)
print(highway_k_dist)
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np

def run_dbscan_auto(data_scaled, dataset_name):
    eps_values = np.linspace(0.5, 5.0, 10)  # تجربة eps من 0.5 إلى 5
    min_samples_values = [3, 5, 8, 10]       # تجربة min_samples
    best_config = None

    for eps in eps_values:
        for min_samples in min_samples_values:
            db = DBSCAN(eps=eps, min_samples=min_samples)
            clusters = db.fit_predict(data_scaled)
            unique_labels = np.unique(clusters)
            n_clusters = len(unique_labels[unique_labels != -1])  # استثناء الضوضاء (-1)

            if n_clusters > 1:
                sil_score = silhouette_score(data_scaled, clusters)
                db_score = davies_bouldin_score(data_scaled, clusters)
                ch_score = calinski_harabasz_score(data_scaled, clusters)
                print(f"{dataset_name} | eps={eps:.2f}, min_samples={min_samples} -> Clusters: {n_clusters}, Silhouette: {sil_score:.3f}")
                best_config = (eps, min_samples, clusters, sil_score, db_score, ch_score)

    if best_config:
        eps, min_samples, clusters, sil_score, db_score, ch_score = best_config
        print("\n=== Best DBSCAN configuration ===")
        print(f"{dataset_name} | eps={eps:.2f}, min_samples={min_samples}")
        print(f"Silhouette Score: {sil_score:.3f}")
        print(f"Davies-Bouldin Index: {db_score:.3f}")
        print(f"Calinski-Harabasz Score: {ch_score:.3f}")
        unique, counts = np.unique(clusters, return_counts=True)
        print("Cluster distribution:", dict(zip(unique, counts)))
        return clusters
    else:
        print(f"No valid clustering found for {dataset_name}")
        return None

# تطبيق على الطرق السريعة
highway_db_clusters = run_dbscan_auto(highway_scaled, "Highway")

# تطبيق على التقاطعات
# intersection_db_clusters = run_dbscan_auto(intersection_scaled, "Intersection")
# تحليل بصري باستخدام PCA
def plot_pca_clusters(data_scaled, clusters, title):
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(data_scaled)

    df_pca = pd.DataFrame()
    df_pca['PCA1'] = pca_result[:, 0]
    df_pca['PCA2'] = pca_result[:, 1]
    df_pca['Cluster'] = clusters

    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_pca, x='PCA1', y='PCA2', hue='Cluster', palette='Set2', s=50)
    plt.title(f' PCA  - {title}')
    plt.show()
# -----------------------------
# تحليل بصري باستخدام PCA
plot_pca_clusters(highway_scaled, highway_db_clusters, "DBSCAN - Highways")
# plot_pca_clusters(intersection_scaled, intersection_db_clusters, "DBSCAN - Intersections")
def plot_pca(driver_scaled, clusters, driver_id, environment):
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(driver_scaled)
    df_pca = pd.DataFrame({
        'PCA1': pca_result[:,0],
        'PCA2': pca_result[:,1],
        'Cluster': clusters
    })
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=df_pca, x='PCA1', y='PCA2', hue='Cluster', palette='Set2', s=50)
    plt.title(f'PCA Clustering for Driver {driver_id} - {environment}')
    plt.show()

# تحليل كل سائق
drivers = data['Class'].unique()

for driver in drivers:
    for env_name, env_data in [('Highway', highway_data), ('Intersection', intersection_data)]:
        driver_data = env_data[env_data['Class'] == driver]
        if driver_data.empty:
            continue

        driver_features = driver_data[important_features].fillna(0)
        driver_scaled = scaler.fit_transform(driver_features)

        # KMeans clustering
        kmeans = KMeans(n_clusters=3, random_state=42)
        clusters = kmeans.fit_predict(driver_scaled)
        driver_data['Cluster'] = clusters

        # مؤشرات التقييم
        silhouette = silhouette_score(driver_scaled, clusters)
        db_index = davies_bouldin_score(driver_scaled, clusters)
        ch_score = calinski_harabasz_score(driver_scaled, clusters)

        print(f"\nDriver {driver} - {env_name}:")
        print(f"Silhouette Score: {silhouette:.3f}")
        print(f"Davies-Bouldin Index: {db_index:.3f}")
        print(f"Calinski-Harabasz Score: {ch_score:.3f}")
        print(f"Cluster Distribution: {dict(zip(*np.unique(clusters, return_counts=True)))}")

        # رسم PCA
        plot_pca(driver_scaled, clusters, driver, env_name)


In [ ]:
##data3
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob
base_path = '/content/drive/MyDrive/UAH-DRIVESET-v1'

# البحث عن جميع ملفات SEMANTIC_FINAL.txt
semantic_files = glob.glob(base_path + '/**/SEMANTIC_FINAL.txt', recursive=True)

# قائمة لتجميع البيانات
all_data = []

for file in semantic_files:
    try:
        # قراءة الملف كقائمة من القيم الرقمية
        with open(file, 'r') as f:
            lines = f.readlines()

        # تأكد أن عدد القيم كافٍ
        if len(lines) < 54:
            print(f" الملف أقل من 54 سطر: {file}")
            continue

        # استخراج القيم المطلوبة
        numeric_values = []
        for line in lines:
            try:
                numeric_values.append(float(line.strip().split()[0]))
            except:
                numeric_values.append(None)

        # تأكيد وجود القيم المطلوبة (نستخدم أول 54 فقط)
        numeric_values = numeric_values[:54]

        # استخراج السائق والسلوك ونوع الطريق من اسم المجلد
        parts = file.split('/')
        trip_folder = parts[-2]  # مثال: 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
        driver = parts[-3]

        trip_parts = trip_folder.split('-')
        behavior = trip_parts[3] if len(trip_parts) > 3 else "UNKNOWN"
        road_type = trip_parts[4] if len(trip_parts) > 4 else "UNKNOWN"
        distance = trip_parts[1].replace('km', '') if 'km' in trip_parts[1] else None

        # بناء السجل
        data_entry = {
            'Driver': driver,
            'Trip': trip_folder,
            'Behavior': behavior.replace("NORMAL1", "NORMAL").replace("NORMAL2", "NORMAL"),
            'RoadType': road_type,
            'DistanceKM': float(distance) if distance else None,
        }

        # إضافة الـ 54 سمة كـ Attr_1 إلى Attr_54
        for i in range(54):
            data_entry[f'Attr_{i+1}'] = numeric_values[i]

        all_data.append(data_entry)

    except Exception as e:
        print(f" خطأ في قراءة الملف {file}: {e}")

# تحويل البيانات إلى DataFrame
df = pd.DataFrame(all_data)
print(" Sample data:")
print(df.head())

# معلومات السائقين الديموغرافية
driver_info = {
    'D1': {'Gender': 'Male', 'AgeRange': '40-50', 'FuelType': 'Diesel', 'VehicleModel': 'Audi Q5 (2014)'},
    'D2': {'Gender': 'Male', 'AgeRange': '20-30', 'FuelType': 'Diesel', 'VehicleModel': 'Mercedes B180 (2013)'},
    'D3': {'Gender': 'Male', 'AgeRange': '20-30', 'FuelType': 'Diesel', 'VehicleModel': 'Citroën C4 (2015)'},
    'D4': {'Gender': 'Female', 'AgeRange': '30-40', 'FuelType': 'Gasoline', 'VehicleModel': 'Kia Picanto (2004)'},
    'D5': {'Gender': 'Male', 'AgeRange': '30-40', 'FuelType': 'Gasoline', 'VehicleModel': 'Opel Astra (2007)'},
    'D6': {'Gender': 'Male', 'AgeRange': '40-50', 'FuelType': 'Electric', 'VehicleModel': 'Citroën C-Zero (2011)'},
}

driver_df = pd.DataFrame.from_dict(driver_info, orient='index').reset_index()
driver_df.rename(columns={'index': 'Driver'}, inplace=True)

# إضافة متوسط العمر (لتحليل عددي)
driver_df['AgeMid'] = driver_df['AgeRange'].apply(lambda x: sum(map(int, x.split('-'))) / 2)

# دمج مع بيانات السائقين الرئيسية
df = df.merge(driver_df, on='Driver', how='left')

# -----------------------------
# تجهيز البيانات للـ ML
# -----------------------------
df['Behavior'] = df['Behavior'].replace({
    "NORMAL1": "NORMAL",
    "NORMAL2": "NORMAL",
    "DROWSY": "CONSERVATIVE"  # استبدال DROWSY بـ CONSERVATIVE
})

# -----------------------------
# تحويل الأعمدة النصية إلى أرقام
# -----------------------------
le_behavior = LabelEncoder()
df['BehaviorEncoded'] = le_behavior.fit_transform(df['Behavior'])

le_gender = LabelEncoder()
df['GenderEncoded'] = le_gender.fit_transform(df['Gender'])

le_road = LabelEncoder()
df['RoadTypeEncoded'] = le_road.fit_transform(df['RoadType'])

# -----------------------------
# اختيار الميزات
# -----------------------------
feature_cols = [f'Attr_{i}' for i in range(1,55)] + ['Gender', 'AgeMid', 'RoadType']
X = df[feature_cols].copy()
y = df['BehaviorEncoded']

# -----------------------------
# تحويل الأعمدة النصية إلى أرقام للـ ML
X_encoded = X.copy()
X_encoded['Gender'] = le_gender.transform(X_encoded['Gender'])
X_encoded['RoadType'] = le_road.transform(X_encoded['RoadType'])

# ملء القيم الفارغة
X_encoded = X_encoded.fillna(X_encoded.mean())

# تطبيع الميزات
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

from scipy.stats import f_oneway

# تقسيم العمر حسب سلوك القيادة
groups = df.groupby('Behavior')['AgeMid'].apply(list)

f_stat, p_value = f_oneway(*groups)

print("ANOVA Age vs Behavior:")
print("F-statistic =", f_stat)
print("p-value =", p_value)
from scipy.stats import chi2_contingency

# جدول تقاطعي
contingency_table = pd.crosstab(df['Gender'], df['Behavior'])

chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square Gender vs Behavior:")
print("Chi2 =", chi2)
print("p-value =", p)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.boxplot(x='Behavior', y='AgeMid', data=df)
plt.title('Age Distribution by Driving Behavior')
plt.show()
sns.countplot(x='Behavior', hue='Gender', data=df)
plt.title('Gender Distribution across Driving Behaviors')
plt.show()
for road in df['RoadType'].unique():
    subset = df[df['RoadType'] == road]

    # ANOVA
    groups = subset.groupby('Behavior')['AgeMid'].apply(list)
    if len(groups) > 1:
        f, p = f_oneway(*groups)
        print(f"\nRoad: {road}")
        print("Age p-value:", p)

In [ ]:
##data4

import os
import shutil
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm

dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Database (1)/'
features_path = os.path.join(dataset_path, 'E4 (1)/')
label_path = os.path.join(dataset_path, 'Subj_metric (1)/')
processed_dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/'
model_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Models/'

# إنشاء المجلدات إذا لم تكن موجودة
os.makedirs(processed_dataset_path, exist_ok=True)
os.makedirs(model_path, exist_ok=True)

# فك ضغط ملفات E4
def process_features_folders(base_directory, processed_dataset_path):
    for serial_number in tqdm(range(1, 14), desc="Processing E4 folders"):
        folder_name = f'{serial_number}-E4-Drv{serial_number} (1)'
        folder_path = os.path.join(base_directory, folder_name)

        if os.path.exists(folder_path) and os.path.isdir(folder_path):
            files_in_folder = os.listdir(folder_path)
            for file_name in tqdm(files_in_folder, desc="Unzipping files", leave=False):
                file_path = os.path.join(folder_path, file_name)
                if file_name.endswith('.zip'):
                    new_folder_name = f'{serial_number}_unzipped'
                    new_folder_path = os.path.join(processed_dataset_path, new_folder_name)
                    os.makedirs(new_folder_path, exist_ok=True)
                    with zipfile.ZipFile(file_path, 'r') as zip_ref:
                        zip_ref.extractall(new_folder_path)
        else:
            tqdm.write(f"Folder {folder_name} not found.")

#  نسخ ملفات Subj_metric
def process_label_files(base_directory, processed_dataset_path):
    for serial_number in tqdm(range(1, 14), desc="Processing label files"):
        file_name = f'SM_Drv{serial_number} (1).csv'
        file_path = os.path.join(base_directory, file_name)
        if os.path.exists(file_path) and os.path.isfile(file_path):
            new_folder_name = f'{serial_number}_unzipped'
            new_folder_path = os.path.join(processed_dataset_path, new_folder_name)
            os.makedirs(new_folder_path, exist_ok=True)
            new_file_path = os.path.join(new_folder_path, file_name)
            shutil.copy(file_path, new_file_path)
        else:
            tqdm.write(f"File {file_path} not found.")

#  معالجة بيانات معدل ضربات القلب
def process_hr_data(base_path):
    for serial_number in tqdm(range(1, 14), desc="Processing HR Data"):
        folder_name = f"{serial_number}_unzipped"
        current_folder_path = os.path.join(base_path, folder_name)
        if os.path.exists(current_folder_path) and os.path.isdir(current_folder_path):
            hr_data_path = os.path.join(current_folder_path, "HR.csv")
            try:
                hr_data = np.loadtxt(hr_data_path, delimiter=',')
                hr_data = np.repeat(hr_data, 4)  # تكرار البيانات لتطابق الطول
                np.savetxt(os.path.join(current_folder_path, "HR_new.csv"), hr_data, delimiter=',')
            except Exception as e:
                print(f"Error processing HR data in {folder_name}: {e}")

def dataframe_one_folder(serial_number, base_path):
    folder_name = f"{serial_number}_unzipped"
    current_folder_path = os.path.join(base_path, folder_name)

    if os.path.exists(current_folder_path) and os.path.isdir(current_folder_path):

        eda = pd.read_csv(
            os.path.join(current_folder_path, "EDA.csv"),
            header=2,
            names=['EDA']
        )

        hr = pd.read_csv(
            os.path.join(current_folder_path, "HR_new.csv"),
            header=12,
            names=['HR']
        )

        temp = pd.read_csv(
            os.path.join(current_folder_path, "TEMP.csv"),
            header=2,
            names=['TEMP']
        )

        stress = pd.read_csv(
            os.path.join(current_folder_path, f"SM_Drv{serial_number} (1).csv"),
            header=1,
            names=['STRESS']
        )

        stress = stress * 10  # تعديل وحدة STRESS

        min_len = min(len(eda), len(hr), len(temp), len(stress))

        eda = eda.iloc[:min_len, 0]
        hr = hr.iloc[:min_len, 0]
        temp = temp.iloc[:min_len, 0]
        stress = stress.iloc[:min_len, 0]

        df_original = pd.concat([eda, hr, temp, stress], axis=1)
# خريطة الـ Drives إلى السائق الحقيقي
        driver_mapping = {
            1: 'D1',
            2: 'D2',
            3: 'D3',
            4: 'D4',
            5: 'D5',
            6: 'D2',   # second drive of D2
            7: 'D6',
            8: 'D1',   # second drive of D1
            9: 'D7',
            10: 'D8',
            11: 'D9',
            12: 'D1',  # third drive of D1
            13: 'D8'   # second drive of D8
        }
        # إضافة عمود السائق
# إضافة عمود السائق الحقيقي
        df_original["Drive_ID"] = serial_number

        df_original['Driver_ID'] = driver_mapping[serial_number]
        return df_original


#  دمج كل بيانات السائقين وحفظ CSV واحد
def combined_csv(base_path):
    df_combined = pd.DataFrame()
    for serial_number in tqdm(range(1, 14), desc="Combining all folders"):
        df_original = dataframe_one_folder(serial_number, base_path)
        df_combined = pd.concat([df_combined, df_original], ignore_index=True)
    df_combined.to_csv(os.path.join(base_path, 'df_combined_data.csv'), index=False)
    print("CSV File saved successfully: ", df_combined.shape)



process_features_folders(features_path, processed_dataset_path)
process_label_files(label_path, processed_dataset_path)
process_hr_data(processed_dataset_path)
combined_csv(processed_dataset_path)
# ==============================
# استيراد المكتبات
# ==============================
from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis
from tqdm import tqdm
# ==============================
# مسار CSV المدمج من الخطوة السابقة
# ==============================
processed_dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/'
csv_path = f'{processed_dataset_path}df_combined_data.csv'

# قراءة CSV
df_combined = pd.read_csv(csv_path)
print("Combined CSV shape:", df_combined.shape)

# ==============================
# دوال استخراج الميزات
# ==============================
def statistical_features(arr):
    vmin = np.amin(arr)
    vmax = np.amax(arr)
    mean = np.mean(arr)
    std = np.std(arr)
    return vmin, vmax, mean, std

def shape_features(arr):
    skewness = skew(arr)
    kurt_val = kurtosis(arr)
    return skewness, kurt_val

def calculate_rms(signal):
    diff_squared = np.square(np.ediff1d(signal))
    rms_value = np.sqrt(np.mean(diff_squared))
    return rms_value

def extract_features(data, window_size=40, step=20):
    cols = [
        'EDA_Mean', 'EDA_Min', 'EDA_Max', 'EDA_Std', 'EDA_Kurtosis', 'EDA_Skew',
        'EDA_Num_Peaks', 'EDA_Amplitude', 'EDA_Duration',
        'HR_Mean', 'HR_Min', 'HR_Max', 'HR_Std', 'HR_RMS',
        'TEMP_Mean', 'TEMP_Min', 'TEMP_Max', 'TEMP_Std', 'STRESS','Driver_ID'
    ]

    df_features = pd.DataFrame(columns=cols)
    index = 0

    for i in tqdm(range(0, len(data['EDA']), step), desc="Processing rows"):
        df_partial = data.iloc[i:i+window_size]
        if len(df_partial) < window_size:
            continue

        eda = df_partial['EDA'].values
        hr = df_partial['HR'].values
        temp = df_partial['TEMP'].values
        stress = df_partial['STRESS'].values
        driver_id = df_partial['Driver_ID'].mode()[0]
        eda_min, eda_max, eda_mean, eda_std = statistical_features(eda)
        hr_min, hr_max, hr_mean, hr_std = statistical_features(hr)
        temp_min, temp_max, temp_mean, temp_std = statistical_features(temp)
        stress_min, stress_max, stress_mean, stress_std = statistical_features(stress)
        eda_skew, eda_kurtosis = shape_features(eda)

        hr_rms = calculate_rms(hr)
        temp_rms = calculate_rms(temp)

        peaks, properties = find_peaks(eda, width=5)
        num_peaks = len(peaks)
        if len(peaks) > 0:
            prominences = np.array(properties['prominences'])
            widths = np.array(properties['widths'])
            amplitude = np.sum(prominences)
            duration = np.sum(widths)
        else:
            amplitude = 0
            duration = 0

        df_features.loc[index] = [
            eda_mean, eda_min, eda_max, eda_std, eda_kurtosis, eda_skew, num_peaks,
            amplitude, duration,
            hr_mean, hr_min, hr_max, hr_std, hr_rms,
            temp_mean, temp_min, temp_max, temp_std, stress_mean,driver_id
        ]

        index += 1

    return df_features

# ==============================
# استخراج الميزات
# ==============================
df_features = extract_features(df_combined)
print("Features extracted shape:", df_features.shape)

# ==============================
# توليد ميزات Lag
# ==============================
def generate_lag_features(input_df, columns, lags):
    lag_df = pd.DataFrame()
    for col in tqdm(columns, desc="Generating lag features"):
        for lag in lags:
            lag_df[f'{col}_lag{lag}'] = input_df[col].shift(lag)
    lag_df = lag_df.fillna(0)
    return lag_df

cols_to_lag = ['HR_Mean', 'TEMP_Mean', 'EDA_Mean']
lags = list(range(10, 0, -1))  # 10 إلى 1
df_lag_features = generate_lag_features(df_features, cols_to_lag, lags)
print("Lag features shape:", df_lag_features.shape)

# ==============================
# دمج كل الميزات في DataFrame واحد
# ==============================
df_total = pd.concat([df_lag_features, df_features.reset_index(drop=True)], axis=1)
print("Total DataFrame shape:", df_total.shape)

# ==============================
# حفظ CSV نهائي للميزات
# ==============================
final_csv_path = f'{processed_dataset_path}df_features_total.csv'
df_total.to_csv(final_csv_path, index=False)
print("Final features CSV saved:", final_csv_path)

from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis
from tqdm import tqdm
import os

# ==============================
# مسارات البيانات
# ==============================
processed_dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/'
sessions = [f'{i}_unzipped' for i in range(1, 14)]  # جلسات 1 إلى 13
combined_csv_path = f'{processed_dataset_path}df_combined_data.csv'

# ==============================
# قراءة CSV المدمج الأساسي
# ==============================
df_combined = pd.read_csv(combined_csv_path)
print("Combined CSV shape:", df_combined.shape)

# ==============================
# دوال استخراج الميزات
# ==============================
def statistical_features(arr):
    return np.amin(arr), np.amax(arr), np.mean(arr), np.std(arr)

def shape_features(arr):
    return skew(arr), kurtosis(arr)

def calculate_rms(signal):
    diff_squared = np.square(np.ediff1d(signal))
    return np.sqrt(np.mean(diff_squared))

# ==============================
# دالة لاستخراج ميزات HR, EDA, TEMP, STRESS, ACC
# ==============================
def extract_features(data, acc_data=None, window_size=40, step=20):
    cols = [
        'EDA_Mean','EDA_Min','EDA_Max','EDA_Std','EDA_Kurtosis','EDA_Skew',
        'EDA_Num_Peaks','EDA_Amplitude','EDA_Duration',
        'HR_Mean','HR_Min','HR_Max','HR_Std','HR_RMS',
        'TEMP_Mean','TEMP_Min','TEMP_Max','TEMP_Std',
        'STRESS',
        'AccX_Mean','AccX_Std','AccX_RMS',
        'AccY_Mean','AccY_Std','AccY_RMS',
        'AccZ_Mean','AccZ_Std','AccZ_RMS','driver_id','Drive_ID'
    ]
    df_features = pd.DataFrame(columns=cols)
    index = 0
    if acc_data is not None:
        min_len = min(len(data), len(acc_data))
        data = data.iloc[:min_len].reset_index(drop=True)
        acc_data = acc_data.iloc[:min_len].reset_index(drop=True)

    for i in tqdm(range(0, len(data['EDA']), step), desc="Processing rows"):
        df_partial = data.iloc[i:i+window_size]
        if len(df_partial) < window_size:
            continue

        # ==============================
        # بيانات فسيولوجية
        # ==============================
        eda = df_partial['EDA'].values
        hr = df_partial['HR'].values
        temp = df_partial['TEMP'].values
        stress = df_partial['STRESS'].values
        driver_id = df_partial['Driver_ID'].mode()[0]
        drive_id = df_partial['Drive_ID'].mode()[0]
        eda_min, eda_max, eda_mean, eda_std = statistical_features(eda)
        hr_min, hr_max, hr_mean, hr_std = statistical_features(hr)
        temp_min, temp_max, temp_mean, temp_std = statistical_features(temp)
        stress_mean = np.mean(stress)

        eda_skew, eda_kurtosis = shape_features(eda)
        hr_rms = calculate_rms(hr)
        temp_rms = calculate_rms(temp)

        peaks, properties = find_peaks(eda, width=5)
        num_peaks = len(peaks)
        amplitude = np.sum(properties['prominences']) if len(peaks)>0 else 0
        duration = np.sum(properties['widths']) if len(peaks)>0 else 0

        # ==============================
        # ميزات ACC
        # ==============================
        if acc_data is not None:
            acc_partial = acc_data.iloc[i:i+window_size]
            accX = acc_partial['AccX'].values
            accY = acc_partial['AccY'].values
            accZ = acc_partial['AccZ'].values

            accX_mean, _, _, accX_std = statistical_features(accX)
            accX_rms = calculate_rms(accX)
            accY_mean, _, _, accY_std = statistical_features(accY)
            accY_rms = calculate_rms(accY)
            accZ_mean, _, _, accZ_std = statistical_features(accZ)
            accZ_rms = calculate_rms(accZ)
        else:
            accX_mean = accX_std = accX_rms = 0
            accY_mean = accY_std = accY_rms = 0
            accZ_mean = accZ_std = accZ_rms = 0

        # ==============================
        # حفظ الصف
        # ==============================
        df_features.loc[index] = [
            eda_mean, eda_min, eda_max, eda_std, eda_kurtosis, eda_skew, num_peaks,
            amplitude, duration,
            hr_mean, hr_min, hr_max, hr_std, hr_rms,
            temp_mean, temp_min, temp_max, temp_std,
            stress_mean,
            accX_mean, accX_std, accX_rms,
            accY_mean, accY_std, accY_rms,
            accZ_mean, accZ_std, accZ_rms,driver_id, drive_id
        ]
        index += 1

    return df_features

# ==============================
# دمج ملفات ACC لجميع الجلسات
# ==============================
all_acc = []
for sess in sessions:
    acc_path = os.path.join(processed_dataset_path, sess, 'ACC.csv')
    df_acc = pd.read_csv(acc_path, skiprows=2, header=None, names=['AccX','AccY','AccZ'])
    all_acc.append(df_acc)

df_all_acc = pd.concat(all_acc, ignore_index=True)
print("Combined ACC shape:", df_all_acc.shape)

# ==============================
# استخراج الميزات مع ACC
# ==============================
df_features = extract_features(df_combined, acc_data=df_all_acc)
print("Features extracted shape:", df_features.shape)

# ==============================
# توليد ميزات Lag
# ==============================
def generate_lag_features(input_df, columns, lags):
    lag_df = pd.DataFrame()
    for col in tqdm(columns, desc="Generating lag features"):
        for lag in lags:
            lag_df[f'{col}_lag{lag}'] = input_df[col].shift(lag)
    return lag_df.fillna(0)

cols_to_lag = ['HR_Mean','TEMP_Mean','EDA_Mean','AccX_Mean','AccY_Mean','AccZ_Mean']
lags = list(range(10,0,-1))
df_lag_features = generate_lag_features(df_features, cols_to_lag, lags)
print("Lag features shape:", df_lag_features.shape)

# ==============================
# دمج كل الميزات في DataFrame واحد
# ==============================
df_total = pd.concat([df_lag_features, df_features.reset_index(drop=True)], axis=1)
print("Total DataFrame shape:", df_total.shape)

# ==============================
# حفظ CSV نهائي للميزات
# ==============================
final_csv_path = os.path.join(processed_dataset_path, 'df_features_total_with_ACC.csv')
df_total.to_csv(final_csv_path, index=False)
print("Final features CSV saved:", final_csv_path)

from sklearn.preprocessing import StandardScaler

X = df.drop(columns=[]).select_dtypes(include='number')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in range(2,8):
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)
    print(k, score)
kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)
cluster_stats = df.groupby('Cluster')[
    ['HR_Mean','EDA_Mean','STRESS','AccX_Mean']
].mean()

print(cluster_stats)


behavior_map = {
    1: "Conservative",
    0: "Normal",
    2: "Aggressive"
}

df["Behavior_Label"] = df["Cluster"].map(
    behavior_map
)

print(
    df[["Cluster","Behavior_Label"]].head()
)
output_path = f'{processed_dataset_path}df_clustered_behavior_labeled.csv'
df.to_csv(output_path, index=False)
print(f"\n✅ Final clustered file saved with labels: {output_path}")
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=df['Cluster']
)

plt.show()
# data4

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# ==============================
# 📂 تحميل البيانات
# ==============================
processed_dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/'
file_path = f'{processed_dataset_path}df_clustered_behavior_labeled.csv'

df = pd.read_csv(file_path)
print("✅ Data loaded:", df.shape)

# ==============================
# ⚙️ حساب Acc_Magnitude لو غير موجود
# ==============================
if 'Acc_Magnitude' not in df.columns:
    df['Acc_Magnitude'] = np.sqrt(df['AccX_Mean']**2 + df['AccY_Mean']**2 + df['AccZ_Mean']**2)

# ==============================
# 🎯 اختيار الميزات والهدف
# ==============================
features = [
    'HR_Mean','EDA_Mean','TEMP_Mean',
    'AccX_Mean','AccY_Mean','AccZ_Mean',
    'STRESS','Acc_Magnitude'
]

target = 'Behavior_Label'

# ترميز الفئات إلى أرقام
le = LabelEncoder()
df[target] = le.fit_transform(df[target])

X = df[features].values
y = df[target].values

# ==============================
# 🔹 تقسيم البيانات
# ==============================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# عدد العينات لكل مجموعة
print("عدد عينات Training set:", X_train.shape[0])
print("عدد عينات Testing set:", X_test.shape[0])
# ==============================
# 🎯 التحضير والتحميل
# ==============================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout

# ==============================
# 📂 تحميل البيانات
# ==============================
processed_dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/'
file_path = f'{processed_dataset_path}df_clustered_behavior_labeled.csv'

df = pd.read_csv(file_path)
print("✅ Data loaded:", df.shape)
target = 'Behavior_Label'

le = LabelEncoder()
df[target] = le.fit_transform(df[target])

# ==============================
# 🎯 تحديد الميزات
# ==============================
all_features = df.select_dtypes(include=[np.number]).columns.tolist()
all_features.remove(target)
if 'Cluster' in all_features:
    all_features.remove('Cluster')

X = df[all_features].values
y = df[target].values

# ==============================
# 🔹 تقسيم وتقييس البيانات
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==============================
# 🔹 دالة لتقييم النماذج
# ==============================
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    print(f"\n--- {model_name} ---")
    print("Accuracy :","{:.4f}".format( accuracy_score(y_test, y_pred)))
    print("Precision:", "{:.4f}".format(precision_score(y_test, y_pred, average='weighted')))
    print("Recall   :","{:.4f}".format( recall_score(y_test, y_pred, average='weighted')))
    print("F1 Score :","{:.4f}".format( f1_score(y_test, y_pred, average='weighted')))

    # ROC-AUC requires probability scores
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
        try:
            roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
            print("ROC-AUC :","{:.4f}".format( roc_auc))
        except:
            print("ROC-AUC : Not applicable for this dataset")
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm,
                annot=True,
                fmt='d',
                cmap='Blues',
                xticklabels=le.classes_,
                yticklabels=le.classes_,
                annot_kws={"size": 18, "weight": "bold"})  # 🔥 تكبير الأرقام

    plt.title(f"{model_name} Confusion Matrix ", fontsize=14)
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("Actual", fontsize=12)
    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)
    plt.show()
    return y_pred


# ==============================
# 1️⃣ Random Forest (All Features)
# ==============================
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = evaluate_model(rf, X_test_scaled, y_test, "Random Forest (All Features)")

# ==============================
# 2️⃣ SVM (All Features)
# ==============================
svm = SVC(kernel='rbf', C=5, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = evaluate_model(svm, X_test_scaled, y_test, "SVM (All Features)")

# ==============================
# 3️⃣ MLP (All Features)
# ==============================
mlp = MLPClassifier(hidden_layer_sizes=(128,64), max_iter=500, random_state=42)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = evaluate_model(mlp, X_test_scaled, y_test, "MLP (All Features)")

# ==============================
# 4️⃣ XGBoost (All Features)
# ==============================
xgb_model = xgb.XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = evaluate_model(xgb_model, X_test_scaled, y_test, "XGBoost (All Features)")

# ==============================
# 5️⃣ KNN (All Features)
# ==============================
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = evaluate_model(knn, X_test_scaled, y_test, "KNN (All Features)")

# ==============================
# 6️⃣ 1D-CNN (All Features)
# ==============================
X_train_cnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_cnn = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=len(le.classes_))
y_test_cnn = tf.keras.utils.to_categorical(y_test, num_classes=len(le.classes_))

cnn_model = Sequential([
    Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1],1)),
    Conv1D(16, kernel_size=3, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(len(le.classes_), activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history =cnn_model.fit(X_train_cnn, y_train_cnn, epochs=15, batch_size=32, validation_split=0.1, verbose=1)

loss, acc = cnn_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"\n--- 1D-CNN (All Features) Accuracy: {acc*100:.2f}% ---")
import pandas as pd
df_history = pd.DataFrame(history.history)
df_history['Epoch'] = range(1, len(df_history)+1)

# حساب المتوسط لكل عمود
avg_row = pd.DataFrame(df_history.mean()).T
avg_row['Epoch'] = 'Average'

# دمج المتوسط مع الجدول الأصلي
df_history = pd.concat([df_history, avg_row], ignore_index=True)
print("\n📊 Training History with Average:")
print(df_history)

# ==============================
# 🔹 رسم Accuracy و Loss لكل Epoch
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(df_history['Epoch'][:-1], df_history['accuracy'][:-1], marker='o', label='Train Accuracy')
plt.plot(df_history['Epoch'][:-1], df_history['val_accuracy'][:-1], marker='o', label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('1D-CNN Accuracy per Epoch')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10,5))
plt.plot(df_history['Epoch'][:-1], df_history['loss'][:-1], marker='o', label='Train Loss')
plt.plot(df_history['Epoch'][:-1], df_history['val_loss'][:-1], marker='o', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('1D-CNN Loss per Epoch')
plt.legend()
plt.grid(True)
plt.show()

# ==============================
# 🔹 اختيار أفضل 10 ميزات من RF
# ==============================
importances = rf.feature_importances_
importance_df = pd.DataFrame({'Feature': all_features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
top10_features = importance_df.head(10)['Feature'].tolist()
print("\nTop 10 Features:", top10_features)

# ==============================
# 🔹 تكرار التجربة على Top 10 ميزات
# ==============================

X_top10 = df[top10_features].values
y_top10 = y

X_train, X_test, y_train, y_test = train_test_split(X_top10, y_top10, test_size=0.2, random_state=42, stratify=y_top10)
scaler2 = StandardScaler()
X_train_scaled = scaler2.fit_transform(X_train)
X_test_scaled = scaler2.transform(X_test)

# Random Forest
rf_top = RandomForestClassifier(n_estimators=100, random_state=42)
rf_top.fit(X_train_scaled, y_train)
evaluate_model(rf_top, X_test_scaled, y_test, "Random Forest (Top 10)")

# SVM
svm_top = SVC(kernel='rbf', C=5, gamma='scale', probability=True, random_state=42)
svm_top.fit(X_train_scaled, y_train)
evaluate_model(svm_top, X_test_scaled, y_test, "SVM (Top 10)")

# MLP
mlp_top = MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', solver='adam', max_iter=300, random_state=42)
mlp_top.fit(X_train_scaled, y_train)
evaluate_model(mlp_top, X_test_scaled, y_test, "MLP (Top 10)")

# XGBoost
xgb_top = xgb.XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_top.fit(X_train_scaled, y_train)
evaluate_model(xgb_top, X_test_scaled, y_test, "XGBoost (Top 10)")

# KNN
knn_top = KNeighborsClassifier(n_neighbors=5, metric = 'minkowski')
knn_top.fit(X_train_scaled, y_train)
evaluate_model(knn_top, X_test_scaled, y_test, "KNN (Top 10)")

# 1D-CNN
X_train_cnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_cnn = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))
y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=len(le.classes_))
y_test_cnn = tf.keras.utils.to_categorical(y_test, num_classes=len(le.classes_))

cnn_model = Sequential([
    Conv1D(32, 3, activation='relu', input_shape=(X_train_cnn.shape[1],1)),
    Conv1D(16, 3, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(len(le.classes_), activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = cnn_model.fit(X_train_cnn, y_train_cnn, epochs=15, batch_size=32, validation_split=0.1, verbose=1)

loss, acc = cnn_model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f"\n--- 1D-CNN (Top 10) Accuracy: {acc*100:.2f}% ---")
import pandas as pd
df_history = pd.DataFrame(history.history)
df_history['Epoch'] = range(1, len(df_history)+1)

# حساب المتوسط لكل عمود
avg_row = pd.DataFrame(df_history.mean()).T
avg_row['Epoch'] = 'Average'

# دمج المتوسط مع الجدول الأصلي
df_history = pd.concat([df_history, avg_row], ignore_index=True)
print("\n📊 Training History with Average:")
print(df_history)

# ==============================
# 🔹 رسم Accuracy و Loss لكل Epoch
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(df_history['Epoch'][:-1], df_history['accuracy'][:-1], marker='o', label='Train Accuracy')
plt.plot(df_history['Epoch'][:-1], df_history['val_accuracy'][:-1], marker='o', label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('1D-CNN Accuracy per Epoch')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10,5))
plt.plot(df_history['Epoch'][:-1], df_history['loss'][:-1], marker='o', label='Train Loss')
plt.plot(df_history['Epoch'][:-1], df_history['val_loss'][:-1], marker='o', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('1D-CNN Loss per Epoch')
plt.legend()
plt.grid(True)
plt.show()

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout

# ==============================
# 📂 تحميل البيانات
# ==============================
processed_dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/'
file_path = f'{processed_dataset_path}df_clustered_behavior_labeled.csv'

df = pd.read_csv(file_path)
print("✅ Data loaded:", df.shape)

physio_features = ['HR_Mean','TEMP_Mean','EDA_Mean','STRESS']

mean_stats = df.groupby('Behavior_Label')[physio_features].mean()
print("✅ Mean of physio features by driving behavior:\n")
print(mean_stats)
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

# قائمة الفيتشر الفسيولوجية المهمة
physio_features = ['HR_Mean','TEMP_Mean','EDA_Mean','STRESS']

# رسم وتحليل كل Feature
for feature in physio_features:
    plt.figure(figsize=(6,4))

    # Boxplot لكل سلوك
    sns.boxplot(x='Behavior_Label', y=feature, data=df, palette="Set2")

    # حساب الـ Mean لكل سلوك
    group_mean = df.groupby('Behavior_Label')[feature].mean()

    # رسم النقاط الحمراء للـ Mean
    for i, behavior in enumerate(group_mean.index):
        mean_val = group_mean[behavior]
        plt.scatter(i, mean_val, color='red', s=50, zorder=10, label='Mean' if i==0 else "")

    # ANOVA للتحقق من الفرق الإحصائي بين المجموعات
    groups = [df[df['Behavior_Label']==label][feature] for label in df['Behavior_Label'].unique()]
    stat, p = f_oneway(*groups)

    plt.title(f"{feature} by Driving Behavior\nANOVA p-value = {p}", fontsize=12)
    plt.xlabel("Driving Behavior")
    plt.ylabel(feature)
    plt.grid(True, linestyle='--', alpha=0.3)

    # طباعة النتيجة في الكونسول أيضاً
    print(f"{feature}:")
    print(group_mean)
    print(f"ANOVA F-statistic = {stat:.5f}, p-value = {p}")
    if p < 0.05:
        print("→ Significant difference between driving behaviors.\n")
    else:
        print("→ No significant difference.\n")

    plt.show()

from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout
import tensorflow as tf
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import StandardScaler, LabelEncoder

target = 'Behavior_Label'
le = LabelEncoder()
df[target] = le.fit_transform(df[target])

sequence_features = []

signals = [
    "HR_Mean",
    "EDA_Mean",
    "TEMP_Mean",
    "AccX_Mean",
    "AccY_Mean",
    "AccZ_Mean"
]

for signal in signals:
    for lag in range(10,0,-1):
        sequence_features.append(
            f"{signal}_lag{lag}"
        )

    sequence_features.append(signal)

X_seq = df[sequence_features].values
y = df[target].values
X_seq = X_seq.reshape(
    X_seq.shape[0],
    11,
    6
)
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([

    tf.keras.layers.Input(
        shape=(11,6)
    ),

    LSTM(
        64,
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(
        32,
        activation='relu'
    ),

    Dense(
        len(le.classes_),
        activation='softmax'
    )
])

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
from tensorflow.keras.layers import LSTM

cnn_lstm = Sequential([

    tf.keras.layers.Input(
        shape=(11,6)
    ),

    Conv1D(
        32,
        3,
        activation='relu'
    ),

    Conv1D(
        16,
        3,
        activation='relu'
    ),

    LSTM(
        64
    ),

    Dropout(0.2),

    Dense(
        32,
        activation='relu'
    ),

    Dense(
        len(le.classes_),
        activation='softmax'
    )
])

cnn_lstm.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

# ==========================
# train / test split
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X_seq,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# one-hot encoding
y_train_cat = tf.keras.utils.to_categorical(
    y_train,
    num_classes=len(le.classes_)
)

y_test_cat = tf.keras.utils.to_categorical(
    y_test,
    num_classes=len(le.classes_)
)


# ====================================
# 1️⃣ LSTM
# ====================================
history_lstm = lstm_model.fit(

    X_train,
    y_train_cat,

    epochs=15,
    batch_size=32,

    validation_split=0.1,
    verbose=1
)


# test accuracy + loss
loss_lstm, acc_lstm = lstm_model.evaluate(
    X_test,
    y_test_cat,
    verbose=0
)

print(f"\nLSTM Test Accuracy: {acc_lstm*100:.2f}%")
print(f"LSTM Test Loss    : {loss_lstm:.4f}")


# ====================================
# 2️⃣ CNN-LSTM
# ====================================
history_cnn_lstm = cnn_lstm.fit(

    X_train,
    y_train_cat,

    epochs=15,
    batch_size=32,

    validation_split=0.1,
    verbose=1
)


# test accuracy + loss
loss_cnn, acc_cnn = cnn_lstm.evaluate(
    X_test,
    y_test_cat,
    verbose=0
)

print(f"\nCNN-LSTM Test Accuracy: {acc_cnn*100:.2f}%")
print(f"CNN-LSTM Test Loss    : {loss_cnn:.4f}")


# ====================================
# رسم accuracy و loss
# ====================================

def plot_history(history, model_name):

    df_history = pd.DataFrame(
        history.history
    )

    print(f"\n{model_name} Training History:")
    print(df_history)

    # Accuracy
    plt.figure(figsize=(8,4))
    plt.plot(df_history['accuracy'], marker='o')
    plt.plot(df_history['val_accuracy'], marker='o')

    plt.title(f'{model_name} Accuracy')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(['Train','Validation'])
    plt.grid()
    plt.show()


    # Loss
    plt.figure(figsize=(8,4))
    plt.plot(df_history['loss'], marker='o')
    plt.plot(df_history['val_loss'], marker='o')

    plt.title(f'{model_name} Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(['Train','Validation'])
    plt.grid()
    plt.show()


plot_history(history_lstm, "LSTM")
plot_history(history_cnn_lstm, "CNN-LSTM")


In [ ]:
##data5
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

np.random.seed(42)


processed_dataset_path = '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/'
file_path = f'{processed_dataset_path}df_clustered_behavior_labeled.csv'

real = pd.read_csv(file_path)


real["Acc_Magnitude"] = np.sqrt(
    real["AccX_Mean"]**2 +
    real["AccY_Mean"]**2 +
    real["AccZ_Mean"]**2
)



signals = [
    "HR_Mean",
    "EDA_Mean",
    "TEMP_Mean",
    "AccX_Mean",
    "AccY_Mean",
    "AccZ_Mean",


]

lag_features = []

for signal in signals:
    for lag in range(10,0,-1):
        lag_features.append(
            f"{signal}_lag{lag}"
        )

drivers = sorted(
    real["driver_id"].unique()
)

print("Drivers:", drivers)

driver_profiles = {}

for driver in drivers:

    driver_profiles[driver] = {

        "Age": np.random.randint(
            18,
            70
        ),

        "Gender": np.random.choice(
            ["Male","Female"],
             p=[0.6,0.4]
        ),




    }


n_samples = 30000

rows = []

for i in range(n_samples):

    driver = np.random.choice(
        drivers
    )
    road = np.random.choice(["Highway","Intersection"])

    driver_rows = real[
        real["driver_id"] == driver
    ]

    sample_row = driver_rows.sample(
        1
    ).iloc[0]

    profile = driver_profiles[
        driver
    ]

    age = profile["Age"]
    gender = profile["Gender"]
    road = road
    stress_current = (
    sample_row["STRESS"] +
    np.random.normal(0, real["STRESS"].std() * 0.01)
    )
    seq_dict = {}


    acc_cols = [
        "AccX_Mean",
        "AccY_Mean",
        "AccZ_Mean"
    ]

    acc_mean = driver_rows[
        acc_cols
    ].mean().values

    acc_cov = driver_rows[
        acc_cols
    ].cov().values

    acc_cov += np.eye(3)*1e-6

    acc_current = np.random.multivariate_normal(
        acc_mean,
        acc_cov
    )
    feature_std = real[signals].std()


    for signal in signals:

        values = []

        for lag in range(10,0,-1):

            real_val = sample_row[
                f"{signal}_lag{lag}"
            ]

            noise = np.random.normal(
                0,
                real[signal].std()*0.01
            )

            values.append(
                real_val + noise
            )

        # current value
        if signal=="AccX_Mean":
            current = acc_current[0]

        elif signal=="AccY_Mean":
            current = acc_current[1]

        elif signal=="AccZ_Mean":
            current = acc_current[2]

        else:

            current = (

                sample_row[signal]

                +

                np.random.normal(
                    0,
                    real[signal].std()*0.01
                )
            )

        if road=="Highway":

           if signal in ["AccX_Mean","AccY_Mean","AccZ_Mean"]:
              current += 0.25 * feature_std[signal]

           if signal=="HR_Mean":
              current += 0.15 * feature_std[signal]
           if signal == "STRESS":
              current += 0.10 * feature_std[signal]

        if gender=="Male":

            if "Acc" in signal:
                current += 0.3

        values.append(
            current
        )

        seq_dict[
            signal
        ] = values



    row = {

        "driver_id": driver,

        "Age": age,

        "Gender": gender,

        "Road_Type": road,
        "STRESS": stress_current

    }


    for signal in signals:

        for lag in range(10,0,-1):

            row[
                f"{signal}_lag{lag}"
            ] = seq_dict[
                signal
            ][10-lag]

        row[
            signal
        ] = seq_dict[
            signal
        ][-1]

    rows.append(
        row
    )

df_syn = pd.DataFrame(
    rows
)


df_syn["Acc_Magnitude"] = np.sqrt(

    df_syn["AccX_Mean"]**2 +

    df_syn["AccY_Mean"]**2 +

    df_syn["AccZ_Mean"]**2
)


score = (

    0.25*df_syn["HR_Mean"]

    +

    0.50*df_syn["EDA_Mean"]

    +

    0.15*np.abs(
        df_syn["AccX_Mean"]
    )

    +

    0.05*np.abs(
        df_syn["AccY_Mean"]
    )

    +

    0.05*np.abs(
        df_syn["AccZ_Mean"]
    )
)

q1 = np.quantile(
    score,
    0.30
)

q2 = np.quantile(
    score,
    0.70
)

df_syn["Behavior_Label"] = np.where(

    score < q1,

    "Conservative",

    np.where(

        score > q2,

        "Aggressive",

        "Normal"
    )
)

# ==========================================
# SAVE
# ==========================================
save_path = f"{processed_dataset_path}synthetic_affective_final.csv"

df_syn.to_csv(
    save_path,
    index=False
)

print("\nSaved:")
print(save_path)


print("\nKS TEST")
signals = [
    "HR_Mean",
    "EDA_Mean",
    "TEMP_Mean",
    "AccX_Mean",
    "AccY_Mean",
    "AccZ_Mean",
    "STRESS"


]

for feature in signals:

    ks = ks_2samp(
        real[feature],
        df_syn[feature]
    )

    print(
        f"{feature}: "
        f"D={ks.statistic:.4f}, "
        f"p={ks.pvalue:.4f}"
    )

# correlation
print("\nCORRELATION DIFFERENCE")

corr_real = real[
    signals
].corr()

corr_syn = df_syn[
    signals
].corr()

print(
    (corr_real-corr_syn).abs()
)

print("\nDriving style distribution")

print(
    df_syn["Behavior_Label"]
    .value_counts(normalize=True)
)
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Conv1D, Dense, Dropout


# =============================
# Read data
# =============================
df = pd.read_csv(
    '/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/synthetic_affective_final.csv'
)


# =============================
# Label encoding
# =============================
target = "Behavior_Label"

le = LabelEncoder()

df[target] = le.fit_transform(
    df[target]
)


# =============================
# sequence features
# =============================
sequence_features = []

signals = [
    "HR_Mean",
    "EDA_Mean",
    "TEMP_Mean",
    "AccX_Mean",
    "AccY_Mean",
    "AccZ_Mean"
]


for signal in signals:

    for lag in range(10,0,-1):

        sequence_features.append(
            f"{signal}_lag{lag}"
        )

    sequence_features.append(signal)



# =============================
# plotting function
# =============================
def plot_history(history, title):

    hist = pd.DataFrame(
        history.history
    )


    # Accuracy
    plt.figure(figsize=(7,4))

    plt.plot(
        hist["accuracy"],
        marker='o'
    )

    plt.plot(
        hist["val_accuracy"],
        marker='o'
    )

    plt.title(
        f"{title} Accuracy"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Accuracy")

    plt.legend(
        ["Train","Validation"]
    )

    plt.grid()

    plt.show()


    # Loss
    plt.figure(figsize=(7,4))

    plt.plot(
        hist["loss"],
        marker='o'
    )

    plt.plot(
        hist["val_loss"],
        marker='o'
    )

    plt.title(
        f"{title} Loss"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.legend(
        ["Train","Validation"]
    )

    plt.grid()

    plt.show()



# =============================
# each road separately
# =============================
for road in ["Highway", "Intersection"]:

    print(
        f"\n===== {road} ====="
    )


    df_road = df[
        df["Road_Type"] == road
    ].copy()


    X_seq = df_road[
        sequence_features
    ].values


    y = df_road[
        target
    ].values


    X_seq = X_seq.reshape(
        X_seq.shape[0],
        11,
        6
    )


    X_train, X_test, y_train, y_test = train_test_split(

        X_seq,
        y,

        test_size=0.2,

        random_state=42,

        stratify=y
    )


    y_train_cat = tf.keras.utils.to_categorical(
        y_train
    )

    y_test_cat = tf.keras.utils.to_categorical(
        y_test
    )


    # =================================
    # LSTM
    # =================================
    lstm_model = Sequential([

        tf.keras.layers.Input(
            shape=(11,6)
        ),

        LSTM(
            64
        ),

        Dropout(0.2),

        Dense(
            32,
            activation='relu'
        ),

        Dense(
            len(le.classes_),
            activation='softmax'
        )
    ])


    lstm_model.compile(

        optimizer='adam',

        loss='categorical_crossentropy',

        metrics=['accuracy']
    )


    history_lstm = lstm_model.fit(

        X_train,

        y_train_cat,

        epochs=15,

        batch_size=32,

        validation_split=0.1,

        verbose=1
    )


    loss_lstm, acc_lstm = lstm_model.evaluate(

        X_test,

        y_test_cat,

        verbose=0
    )


    print(
        f"LSTM Accuracy: {acc_lstm:.4f}"
    )


    plot_history(
        history_lstm,
        f"{road} - LSTM"
    )



    # =================================
    # CNN-LSTM
    # =================================
    cnn_lstm = Sequential([

        tf.keras.layers.Input(
            shape=(11,6)
        ),

        Conv1D(
            32,
            3,
            activation='relu'
        ),

        Conv1D(
            16,
            3,
            activation='relu'
        ),

        LSTM(
            64
        ),

        Dropout(0.2),

        Dense(
            32,
            activation='relu'
        ),

        Dense(
            len(le.classes_),
            activation='softmax'
        )
    ])


    cnn_lstm.compile(

        optimizer='adam',

        loss='categorical_crossentropy',

        metrics=['accuracy']
    )


    history_cnn = cnn_lstm.fit(

        X_train,

        y_train_cat,

        epochs=15,

        batch_size=32,

        validation_split=0.1,

        verbose=1
    )


    loss_cnn, acc_cnn = cnn_lstm.evaluate(

        X_test,

        y_test_cat,

        verbose=0
    )


    print(
        f"CNN-LSTM Accuracy: {acc_cnn:.4f}"
    )


    plot_history(
        history_cnn,
        f"{road} - CNN-LSTM"
    )
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             silhouette_score)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.cluster import KMeans

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten, Dropout
df  = pd.read_csv('/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/synthetic_affective_final.csv')

print(df.head())
df  = df.rename(columns={"Behavior_Label": "Driving_Style"})
# =========================
le = LabelEncoder()
df["y"] = le.fit_transform(df["Driving_Style"])

# =========================
# FEATURES (FIXED)
# =========================

# كل الأعمدة الرقمية
features = df.select_dtypes(include=[np.number]).columns.tolist()

# نحذف التارجت الجديد
features.remove("y")



# إذا driver_id كان رقمي
if "driver_id" in all_features:
    all_features.remove("driver_id")

print("Features used:", all_features)

# =========================
# SPLIT DATA
# =========================
X = df[features]
y = df["y"]
# features = [
#     "HR_Mean", "EDA_Mean", "TEMP_Mean",
#     "AccX_Mean", "AccY_Mean", "AccZ_Mean",
#     "STRESS", "Acc_Magnitude"
# ]
df["Gender"] = LabelEncoder().fit_transform(df["Gender"])
features.append("Gender")
df["Road_Type"].value_counts(normalize=True)
road_distribution = df.groupby("Road_Type").size() / len(df)
print(road_distribution)
dist = df.groupby("Road_Type")["Driving_Style"].value_counts(normalize=True).unstack()
print(dist)
pd.crosstab(df["Road_Type"], df["Driving_Style"])
from sklearn.decomposition import PCA

from sklearn.preprocessing import LabelEncoder
le_style = LabelEncoder()
df["Style_Encoded"] = le_style.fit_transform(df["Driving_Style"])
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
road_types = df['Road_Type'].unique()

for road in road_types:
    df_road = df[df['Road_Type'] == road].copy()

    # Standardize features
    X = df_road[features].values
    X_scaled = StandardScaler().fit_transform(X)

    # KMeans لكل طريق
    kmeans = KMeans(n_clusters=3, random_state=42)
    df_road['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

    # PCA للعرض ثنائي
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    df_road['PCA1'] = X_pca[:,0]
    df_road['PCA2'] = X_pca[:,1]

    # مقاييس الكلسترنج
    sil_score = silhouette_score(X_scaled, df_road['KMeans_Cluster'])
    ari_score = adjusted_rand_score(df_road['Driving_Style'], df_road['KMeans_Cluster'])
    nmi_score = normalized_mutual_info_score(df_road['Driving_Style'], df_road['KMeans_Cluster'])

    print(f"\n{road} Clustering Metrics:")
    print(f"Silhouette Score: {sil_score:.4f}")
    print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
    print(f"Normalized Mutual Info (NMI): {nmi_score:.4f}")

    # رسم الكلسترنج
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=df_road, x='PCA1', y='PCA2',
                    hue='KMeans_Cluster', palette='Set2', alpha=0.7)
    plt.title(f"KMeans Clusters on {road} projected on PCA")
    plt.xlabel('PCA1')
    plt.ylabel('PCA2')
    plt.legend(title='Cluster')
    plt.show()

    from scipy.stats import f_oneway

# قائمة الميزات الفسيولوجية
physio_features = ["HR_Mean", "EDA_Mean", "TEMP_Mean",
    "STRESS", ]

anova_results = []

for feature in physio_features:
    # تقسيم البيانات حسب Driving Style
    groups = [group[feature].values for name, group in df.groupby('Driving_Style')]

    # حساب ANOVA
    F, p = f_oneway(*groups)
    anova_results.append({'Feature': feature, 'F-value': F, 'p-value': p})

anova_df = pd.DataFrame(anova_results)
print(anova_df)

# -------------------------------
# رسم متوسطات كل ميزة حسب Driving Style
# -------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))
for feature in physio_features:
    sns.barplot(x='Driving_Style', y=feature, data=df)
    plt.title(f"Mean {feature} per Driving Style")
    plt.show()
from scipy.stats import f_oneway
import matplotlib.pyplot as plt
import seaborn as sns

physio_features = ["HR_Mean", "EDA_Mean", "TEMP_Mean",
    "STRESS", ]
anova_road_results = []

for road in df['Road_Type'].unique():
    print(f"\n--- ANOVA for Road_Type: {road} ---")
    df_road = df[df['Road_Type'] == road]

    for feature in physio_features:
        groups = [group[feature].values for name, group in df_road.groupby('Driving_Style')]
        F, p = f_oneway(*groups)
        anova_road_results.append({
            'Road_Type': road,
            'Feature': feature,
            'F-value': F,
            'p-value': p
        })
        print(f"{feature}: F = {F:.4f}, p = {p:.4e}")


anova_road_df = pd.DataFrame(anova_road_results)

import matplotlib.pyplot as plt
import seaborn as sns

# قائمة الميزات الفسيولوجية
physio_features = ["HR_Mean", "EDA_Mean", "TEMP_Mean",
    "STRESS", ]

# رسم Boxplot لكل ميزة
for feature in physio_features:
    plt.figure(figsize=(8,5))
    sns.boxplot(
        data=df,
        x="Driving_Style",
        y=feature,
        hue="Road_Type",   # يفصل حسب نوع الطريق
        palette="Set2"
    )
    plt.title(f"{feature} Distribution by Driving Style and Road Type")
    plt.ylabel(feature)
    plt.xlabel("Driving Style")
    plt.legend(title="Road Type")
    plt.show()

# -------------------------------
# 1️⃣ تحديد الميزات الفسيولوجية
# -------------------------------

# -------------------------------
# 2️⃣ المتوسط العام لكل أسلوب قيادة
# -------------------------------
mean_overall = df.groupby("Driving_Style")[physio_features].mean().round(2)
print("✅ Mean Values of Physiological Features Across Driving Behavior Categories:")
print(mean_overall)

# -------------------------------
# 3️⃣ المتوسط لكل أسلوب قيادة حسب الطريق
# -------------------------------
mean_by_road = df.groupby(["Road_Type","Driving_Style"])[physio_features].mean().round(2)
print("\n✅ Mean Values per Road Type and Driving Style:")
print(mean_by_road)

# -------------------------------
# 4️⃣ ANOVA لكل ميزة فسيولوجية (كل الطرق معًا)
# -------------------------------
anova_results = []
for feat in physio_features:
    groups = [df[df["Driving_Style"]==style][feat] for style in df["Driving_Style"].unique()]
    F, p = f_oneway(*groups)
    anova_results.append([feat, round(F,3), round(p,6)])

anova_df = pd.DataFrame(anova_results, columns=["Feature", "F-value", "p-value"])
print("\n✅ ANOVA Results Across Driving Styles (All Roads):")
print(anova_df)

# -------------------------------
# 5️⃣ ANOVA لكل ميزة حسب الطريق
# -------------------------------
for road in df["Road_Type"].unique():
    print(f"\n--- ANOVA for Road_Type: {road} ---")
    df_road = df[df["Road_Type"]==road]
    for feat in physio_features:
        groups = [df_road[df_road["Driving_Style"]==style][feat] for style in df_road["Driving_Style"].unique()]
        F, p = f_oneway(*groups)
        print(f"{feat}: F = {F:.4f}, p = {p:.4e}")
X = df[features].values
y = df['Driving_Style'].values

# -------------------------------
# Standardize
# -------------------------------
scaler = StandardScaler()
X = scaler.fit_transform(X)

# -------------------------------
# Split by Road_Type
# -------------------------------
df_highway = df[df['Road_Type']=='Highway']
df_intersection = df[df['Road_Type']=='Intersection']

def get_X_y(df_sub):
    X_sub = df_sub[features].values
    y_sub = le_style.transform(df_sub['Driving_Style'])
    X_sub = scaler.transform(X_sub)
    return X_sub, y_sub

X_h, y_h = get_X_y(df_highway)
X_i, y_i = get_X_y(df_intersection)

# -------------------------------
# Function to evaluate ML models
# -------------------------------
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test) if hasattr(model,'predict_proba') else np.zeros((len(y_test),len(le_style.classes_)))

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    try:
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
    except:
        auc = np.nan
    return acc, prec, rec, f1, auc

# -------------------------------
# Machine Learning Models
# -------------------------------
models_ml = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf',C=5,probability=True,gamma = 'scale'),
    "KNN": KNeighborsClassifier(n_neighbors=5, metric = 'minkowski'),
    "XGBoost": XGBClassifier(n_estimators=100,eval_metric='mlogloss', random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', solver='adam', max_iter=300, random_state=42)
}

results = {}
def plot_cm(model, X_test, y_test, title, labels):
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels,
                yticklabels=labels,
                annot_kws={"size": 18})
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
for road, X_sub, y_sub in [('Highway', X_h, y_h), ('Intersection', X_i, y_i)]:
    print(f"\n--- Evaluating ML models for {road} ---")
    X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.2, random_state=42, stratify=y_sub)

    for name, model in models_ml.items():
        model.fit(X_train, y_train)
        acc, prec, rec, f1, auc = evaluate_model(model, X_test, y_test)
        print(f"{name} | Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | ROC-AUC: {auc:.4f}")
        plot_cm(model, X_test, y_test, f"{name} - {road}", le_style.classes_)

        results[(road,name)] = (acc, prec, rec, f1, auc)

# -------------------------------
# Deep Learning 1D CNN
# -------------------------------

def build_cnn(input_shape, num_classes):
    model = Sequential()
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.3))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


cnn_results = {}
cnn_histories = {}

for road, X_sub, y_sub in [('Highway', X_h, y_h), ('Intersection', X_i, y_i)]:
    print(f"\n--- Training 1D CNN for {road} ---")
    X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.2, random_state=42, stratify=y_sub)

    # إعادة تشكيل البيانات لـ Conv1D
    X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    cnn = build_cnn((X_train_cnn.shape[1], 1), num_classes=len(le_style.classes_))

    # التدريب مع حفظ التاريخ
    history = cnn.fit(X_train_cnn, y_train, epochs=15, batch_size=64, validation_split=0.2, verbose=0)

    # حفظ النتائج
    acc = cnn.evaluate(X_test_cnn, y_test, verbose=0)[1]
    print(f"1D CNN Test Accuracy: {acc:.4f}")
    cnn_results[road] = acc
    cnn_histories[road] = history

    # رسم Accuracy و Loss لكل epoch
    plt.figure(figsize=(12,5))

    # Accuracy
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{road} CNN Accuracy per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Loss
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{road} CNN Loss per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.show()

# -------------------------------
# Feature Importance (Random Forest)
# -------------------------------
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)
importance = rf.feature_importances_
imp_df = pd.DataFrame({'Feature':features,'Importance':importance}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=imp_df)
plt.title("Feature Importance (Random Forest)")
plt.show()

# -------------------------------
# Driver-wise behavior per road
# -------------------------------
driver_road_behavior = df.groupby(['driver_id','Road_Type'])['Driving_Style'].value_counts(normalize=True).unstack().fillna(0)
print(driver_road_behavior)

# -------------------------------
# Statistical Analysis
# -------------------------------
from scipy.stats import f_oneway, chi2_contingency

# Age vs Driving Style
groups_age = [group['Age'].values for name, group in df.groupby('Driving_Style')]
F, p = f_oneway(*groups_age)
print(f"\nANOVA Age vs Driving Style: F = {F:.4f}, p = {p:.4e}")

# Gender vs Driving Style
cont_table = pd.crosstab(df['Gender'], df['Driving_Style'])
chi2, p_gender, dof, expected = chi2_contingency(cont_table)
print(f"Chi-square Gender vs Driving Style: chi2 = {chi2:.4f}, p = {p_gender:.4f}")

# -------------------------------
# Visualization: Driving Style per Road
# -------------------------------
road_style = df.groupby('Road_Type')['Driving_Style'].value_counts(normalize=True).unstack()
road_style.plot(kind='bar', stacked=True, figsize=(8,6))
plt.title("Driving Style Distribution per Road Type")
plt.ylabel("Proportion")
plt.show()


import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1️⃣ Boxplot + Mean Age
# -----------------------------
plt.figure(figsize=(8,6))
ax = sns.boxplot(x='Driving_Style', y='Age', data=df, palette='Set2', showfliers=False)


plt.xlabel("Driving Style")
plt.ylabel("Age (years)")
plt.show()

# -----------------------------
# 2️⃣ Bar plot لنسبة الجنس لكل Driving Style
# -----------------------------
gender_counts = df.groupby(['Driving_Style', 'Gender']).size().unstack(fill_value=0)
gender_props = gender_counts.div(gender_counts.sum(axis=1), axis=0)  # تحويل للنسب

gender_props.plot(kind='bar', stacked=True, figsize=(8,6), colormap='Set2')
plt.ylabel("Proportion")
plt.xlabel("Driving Style")
plt.legend(title='Gender')
plt.show()
gender_counts = df.groupby(
    ['Driving_Style', 'Gender']
).size().unstack(fill_value=0)

# تحويل لنسب
gender_props = gender_counts.div(
    gender_counts.sum(axis=1),
    axis=0
)

# تغيير أسماء الأعمدة
gender_props.columns = ["Female", "Male"]

# الرسم
gender_props.plot(
    kind='bar',
    stacked=True,
    figsize=(8,6),
    colormap='Set2'
)

plt.ylabel("Proportion")
plt.xlabel("Driving Style")
plt.legend(title='Gender')
plt.show()
gender_counts = df.groupby(
    ['Driving_Style', 'Gender']
).size().unstack(fill_value=0)

# تحويل لنسب
gender_props = gender_counts.div(
    gender_counts.sum(axis=1),
    axis=0
)



In [ ]:
##data4->5
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score


# =========================
# LOAD DATA
# =========================
real = pd.read_csv('/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/df_clustered_behavior_labeled.csv')
syn  = pd.read_csv('/content/drive/MyDrive/AffectiveROAD_Data (1)/Processed_Dataset/synthetic_affective_final.csv')

# =========================
# ADD MAGNITUDE (important)
# =========================
for df in [real, syn]:
    df["Acc_Magnitude"] = np.sqrt(
        df["AccX_Mean"]**2 +
        df["AccY_Mean"]**2 +
        df["AccZ_Mean"]**2
    )
print("REAL columns:", real.columns)
print("SYN columns:", syn.columns)

# =========================
# FIX LABEL COLUMN (IMPORTANT)
# =========================

if "Behavior_Label" in real.columns:
    real = real.rename(columns={"Behavior_Label": "Label"})
elif "Label" not in real.columns:
    raise ValueError("Real dataset has no Behavior_Label or Label column")

if "Behavior_Label" in syn.columns:
    syn = syn.rename(columns={"Behavior_Label": "Label"})
elif "Label" not in syn.columns:
    raise ValueError("Synthetic dataset has no Driving_Style or Label column")
# =========================
# LABEL COLUMN UNIFICATION
# =========================
real = real.rename(columns={"Behavior_Label": "Label"})
syn  = syn.rename(columns={"Behavior_Label": "Label"})

# =========================
# ENCODE LABELS (shared space)
# =========================
le = LabelEncoder()

all_labels = pd.concat([real["Label"], syn["Label"]])
le.fit(all_labels)

real["Label"] = le.transform(real["Label"])
syn["Label"]  = le.transform(syn["Label"])

# =========================
# FEATURES (common only)
# =========================
features = [
    "HR_Mean", "EDA_Mean", "TEMP_Mean",
    "AccX_Mean", "AccY_Mean", "AccZ_Mean",
    "STRESS", "Acc_Magnitude"
]

# keep only existing columns
features = [f for f in features if f in real.columns and f in syn.columns]

X_real = real[features].values
y_real = real["Label"].values

X_syn = syn[features].values
y_syn = syn["Label"].values

# =========================
# MODELS
# =========================
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf',C=5,probability=True,gamma = 'scale'),
    "KNN": KNeighborsClassifier(n_neighbors=5, metric = 'minkowski'),
    "XGBoost": XGBClassifier(n_estimators=100,eval_metric='mlogloss', random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', solver='adam', max_iter=300, random_state=42)
}

# =========================
# SCALING FUNCTION
# =========================
def scale_data(X_train, X_test):
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test)

# =========================
# 1) REAL → SYNTHETIC
# =========================
print("\n================ REAL → SYNTHETIC ================\n")

for name, model in models.items():

    Xtr, Xte = scale_data(X_real, X_syn)

    model.fit(Xtr, y_real)
    pred = model.predict(Xte)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_syn, pred))
    print(classification_report(y_syn, pred))

# =========================
# 2) SYNTHETIC → REAL
# =========================
print("\n================ SYNTHETIC → REAL ================\n")

for name, model in models.items():

    Xtr, Xte = scale_data(X_syn, X_real)

    model.fit(Xtr, y_syn)
    pred = model.predict(Xte)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_real, pred))
    print(classification_report(y_real, pred))

# =========================
# 3) MIXED TRAIN TEST (optional but strong for paper)
# =========================
print("\n================ MIXED EXPERIMENT ================\n")

X_all = np.vstack([X_real, X_syn])
y_all = np.hstack([y_real, y_syn])

# shuffle split manually
idx = np.random.permutation(len(X_all))
split = int(0.8 * len(X_all))

train_idx = idx[:split]
test_idx  = idx[split:]

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

X_train, X_test = scale_data(X_train, X_test)

for name, model in models.items():

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, pred))
    print(classification_report(y_test, pred))